# 06 - Popolazione di 10 adulti virtuali e robustezza al rumore del sensore

**Obiettivo.** Estendere il confronto, come richiesto dalla co-relatrice, a ulteriori pazienti virtuali e semi casuali:
1. tutti i 10 adulti di simglucose (`adult#001`-`adult#010`), pasti non annunciati da 60 e 90 g e verifica a digiuno,
   con i moduli attivati separatamente e insieme;
2. 10 semi del rumore del sensore CGM su `adult#001`, `adult#002`, `adult#003` (60 g).

I controllori **non** vengono ritarati sul singolo paziente: cambiano solo le grandezze legate al basale
(limiti dell'integrale e tetto IOB). Il messaggio da verificare è il **compromesso** tra riduzione dell'ipoglicemia
(insulin stacking) e peggioramento del controllo postprandiale, paziente per paziente.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import aid_sim as A, notebook_utils as U
import run_campaign as RC          # stesse funzioni, stessa cache e simulazioni in parallelo della campagna
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 110); pd.set_option("display.max_columns", 40)
SEED = 1                                  # seme del rumore del sensore (uguale per tutti i controllori)
COL = RC.COL
print("Versioni delle librerie usate:"); display(U.library_versions())

Versioni delle librerie usate:


,Pacchetto,Versione
0,Python,3.13.16
1,simglucose,0.2.11
2,numpy,2.4.4
3,scipy,1.17.1
4,pandas,3.0.2
5,matplotlib,3.10.8


## 1. Configurazione
### 1.1 Pazienti (parametri di simglucose)

In [2]:
rows = []
for p in RC.POPOLAZIONE:
    ic = U.initial_conditions(p, SEED)
    rows.append({"Paziente": p, "BW [kg]": round(ic["BW"], 1), "Gb [mg/dL]": round(ic["Gb"], 1),
                 "Basale u_b [U/h]": round(ic["ub"] * 60, 3), "Tetto IOB [U]": round(1.5 * ic["ub"] * 60, 2)})
display(pd.DataFrame(rows))
display(U.scenario_table("adult#001", "0, 60 o 90", SEED).iloc[6:])

,Paziente,BW [kg],Gb [mg/dL],Basale u_b [U/h],Tetto IOB [U]
0,adult#001,102.3,138.6,1.267,1.90
1,adult#002,111.1,136.4,1.370,2.05
2,adult#003,81.6,147.1,1.425,2.14
3,adult#004,63.0,150.7,0.888,1.33
4,adult#005,94.1,142.7,1.180,1.77
5,adult#006,66.1,135.6,1.725,2.59
6,adult#007,91.2,135.3,1.371,2.06
7,adult#008,102.8,143.2,1.141,1.71
8,adult#009,74.6,145.1,1.134,1.70
9,adult#010,73.9,152.8,1.019,1.53


,Voce,Valore,Unita'
6,Seme casuale del sensore,1,-
7,Inizio simulazione,00:00,ora del giorno
8,Assestamento escluso dalle metriche,00:00-08:00,ora del giorno
9,Finestra di analisi,08:00-24:00,ora del giorno
10,Passo di campionamento dt,3.0,min
11,Passi totali / nella finestra,480 / 320,-
12,Glicemia plasmatica iniziale (00:00),138.6,mg/dL
13,Misura CGM iniziale (00:00),152.0,mg/dL
14,"Stati iniziali stimatore IOB (S1, S2)","0.0, 0.0",U
15,Orario del pasto,12:00,ora del giorno


### 1.2 Parametri dei controllori (con unità di misura)

In [3]:
ub = U.initial_conditions("adult#001", SEED)["ub"]
tabs = {n: U.controller_table(A.CFG[n], ub).set_index("Parametro") for n in RC.MODULI}
cmp_ = pd.DataFrame({n: tabs[n]["Valore"].astype(str) for n in RC.MODULI}); cmp_["Unita'"] = tabs["PID"]["Unita'"]
display(cmp_)

,PID,PID cons.,PID+IOB,PID+PLGS,PID+IOB+PLGS,Unita'
Parametro,,,,,,
target,120.0,120.0,120.0,120.0,120.0,mg/dL
kp,0.0001,0.0001,0.0001,0.0001,0.0001,U/min per mg/dL
ki,5e-06,5e-06,5e-06,5e-06,5e-06,U/min per (mg/dL*min)
kd,0.002,0.002,0.002,0.002,0.002,U/min per (mg/dL/min)
n_slope,5,5,5,5,5,campioni
aw_frac,1.0,0.5,1.0,1.0,1.0,adimensionale
aw_frac_low,0.25,0.25,0.25,0.25,0.25,adimensionale
freeze,True,True,True,True,True,-
use_iob,False,False,True,False,True,-


### Unità di misura delle variabili salvate
| Variabile | Unità | Significato |
|---|---|---|
| `t_h` | h | ora del giorno (la simulazione parte alle 00:00) |
| `CGM` | mg/dL | misura del sensore, con rumore: è l'unica grandezza vista dal controllore |
| `BG` | mg/dL | glicemia plasmatica del modello UVa/Padova (Gp/Vg), usata per le metriche |
| `Insulin_Uh` | U/h | velocità di infusione comandata |
| `IOB` | U | insulina attiva netta stimata dal controllore (eccesso rispetto al basale) |
| `Vincolo` | - | vincolo attivo nel passo: nessuno, `iob`, `plgs`, `sat_alta`, `sat_bassa` |

## 2. Popolazione: simulazioni

In [4]:
CARICHI = [0, 60, 90]
keys = [RC.key(p, c, n) for p in RC.POPOLAZIONE for c in CARICHI for n in RC.MODULI]
RC.run_all(keys)
Pop = pd.DataFrame([RC.row(k) for k in keys]).round(1)
print(len(keys), "simulazioni")

150 simulazioni


### 2.1 Sintesi sulla popolazione (media ± deviazione standard sui 10 pazienti)

In [5]:
cols = [RC.COLONNE[c] for c in ["TIR", "TAR", "TBR", "TBR_tardivo", "picco", "IAE_range", "t_rientro", "n_commut", "TV_u"]]
for c in CARICHI:
    print(f"Pasto da {c} g:" if c else "A digiuno (0 g):")
    display(RC.media_std(Pop[Pop["CHO [g]"] == c], ["Controllore"], cols))

A digiuno (0 g):


,Controllore,TIR [%],TAR [%],TBR [%],TBR tardivo [%],Picco [mg/dL],IAE fuori range [mg/dL*h],Rientro in range [h],Commutazioni vincolo [-],Variazione totale infusione [U/h],Non rientra [n]
0,PID,90.6 ± 13.5,0.8 ± 2.7,8.6 ± 12.0,12.9 ± 19.4,150.1 ± 18.4,19.5 ± 33.8,3.4 ± 4.8,0.0 ± 0.0,9.1 ± 1.1,0
1,PID cons.,98.4 ± 4.7,0.2 ± 0.5,1.4 ± 4.2,2.4 ± 7.2,144.5 ± 19.4,1.1 ± 3.6,1.4 ± 3.1,0.0 ± 0.0,8.1 ± 0.8,0
2,PID+IOB,91.7 ± 12.8,0.8 ± 2.6,7.5 ± 11.2,11.9 ± 18.4,149.8 ± 18.4,15.5 ± 29.2,3.4 ± 4.7,14.0 ± 9.1,27.1 ± 11.2,0
3,PID+PLGS,90.0 ± 15.9,3.4 ± 5.8,6.6 ± 10.4,8.3 ± 14.7,159.0 ± 27.2,15.8 ± 30.7,2.1 ± 4.0,8.7 ± 1.8,19.8 ± 4.2,1
4,PID+IOB+PLGS,92.5 ± 14.2,3.2 ± 5.3,4.3 ± 9.2,5.8 ± 12.8,157.6 ± 26.2,10.6 ± 21.4,1.2 ± 3.7,27.6 ± 8.8,46.0 ± 12.9,1


Pasto da 60 g:


,Controllore,TIR [%],TAR [%],TBR [%],TBR tardivo [%],Picco [mg/dL],IAE fuori range [mg/dL*h],Rientro in range [h],Commutazioni vincolo [-],Variazione totale infusione [U/h],Non rientra [n]
0,PID,71.8 ± 15.0,17.0 ± 11.3,11.1 ± 13.1,19.8 ± 23.3,220.7 ± 35.3,131.7 ± 133.0,6.8 ± 2.6,0.0 ± 0.0,9.9 ± 0.5,2
1,PID cons.,75.0 ± 12.2,20.9 ± 10.7,4.1 ± 8.5,7.3 ± 15.1,221.7 ± 32.7,114.5 ± 110.2,6.0 ± 2.3,0.0 ± 0.0,8.5 ± 0.5,0
2,PID+IOB,73.1 ± 15.8,18.8 ± 12.7,8.1 ± 11.3,14.4 ± 20.0,221.0 ± 35.3,128.1 ± 126.3,5.6 ± 2.9,58.8 ± 21.1,107.8 ± 37.1,2
3,PID+PLGS,71.9 ± 10.0,22.3 ± 8.6,5.8 ± 8.0,10.4 ± 14.3,224.4 ± 30.8,118.7 ± 102.2,6.7 ± 2.8,8.6 ± 3.4,19.5 ± 4.7,1
4,PID+IOB+PLGS,73.6 ± 9.5,23.0 ± 9.3,3.4 ± 5.7,6.1 ± 10.1,223.8 ± 30.1,118.4 ± 106.6,6.5 ± 3.0,74.0 ± 18.6,130.6 ± 36.7,0


Pasto da 90 g:


,Controllore,TIR [%],TAR [%],TBR [%],TBR tardivo [%],Picco [mg/dL],IAE fuori range [mg/dL*h],Rientro in range [h],Commutazioni vincolo [-],Variazione totale infusione [U/h],Non rientra [n]
0,PID,61.3 ± 10.8,25.0 ± 9.5,13.7 ± 12.4,24.3 ± 22.0,263.5 ± 51.6,278.5 ± 198.5,6.9 ± 2.1,0.0 ± 0.0,10.3 ± 0.6,5
1,PID cons.,66.4 ± 10.1,28.0 ± 8.2,5.6 ± 9.7,9.9 ± 17.3,268.8 ± 45.4,281.8 ± 181.9,6.7 ± 2.2,0.0 ± 0.0,9.0 ± 0.6,0
2,PID+IOB,63.6 ± 9.8,28.3 ± 9.9,8.1 ± 10.3,14.4 ± 18.3,265.8 ± 51.1,294.2 ± 211.1,7.4 ± 1.8,77.2 ± 23.7,152.4 ± 48.8,2
3,PID+PLGS,64.8 ± 7.7,27.6 ± 7.2,7.7 ± 7.6,13.7 ± 13.6,264.8 ± 43.3,256.8 ± 172.5,8.4 ± 2.9,9.0 ± 3.9,20.8 ± 4.8,3
4,PID+IOB+PLGS,66.3 ± 10.1,30.4 ± 9.7,3.3 ± 7.0,5.8 ± 12.3,266.6 ± 42.8,286.1 ± 192.8,6.1 ± 2.1,91.3 ± 21.4,174.3 ± 50.5,1


### 2.2 Distribuzione delle metriche sui pazienti

In [6]:
MET = ["TIR [%]", "TBR [%]", "Picco [mg/dL]", "IAE fuori range [mg/dL*h]", "Commutazioni vincolo [-]"]
fig, ax = plt.subplots(2, len(MET), figsize=(18, 7))
for i, c in enumerate([60, 90]):
    d = Pop[Pop["CHO [g]"] == c]
    for j, colm in enumerate(MET):
        dati = [d[d.Controllore == n][colm].values for n in RC.MODULI]
        bp = ax[i, j].boxplot(dati, patch_artist=True, widths=0.6)
        for patch, n in zip(bp["boxes"], RC.MODULI): patch.set_facecolor(COL[n]); patch.set_alpha(0.5)
        for k, y in enumerate(dati, 1): ax[i, j].scatter(np.full(len(y), k), y, s=10, color="k", zorder=3)
        ax[i, j].set_xticks(range(1, len(RC.MODULI) + 1)); ax[i, j].set_xticklabels(RC.MODULI, rotation=30, ha="right", fontsize=8)
        ax[i, j].set_title(f"{colm}, {c} g", fontsize=9); ax[i, j].grid(ls=":", alpha=0.5)
plt.tight_layout(); plt.show()

### 2.3 Compromesso paziente per paziente: `PID+IOB+PLGS` meno `PID`

In [7]:
D = []
for c in [60, 90]:
    d = Pop[Pop["CHO [g]"] == c].set_index(["Paziente", "Controllore"])
    for p in RC.POPOLAZIONE:
        a, b = d.loc[(p, "PID")], d.loc[(p, "PID+IOB+PLGS")]
        D.append({"Paziente": p, "CHO [g]": c, **{f"Δ {m}": round(b[m] - a[m], 1) for m in
                  ["TIR [%]", "TAR [%]", "TBR [%]", "TBR tardivo [%]", "Picco [mg/dL]", "IAE fuori range [mg/dL*h]"]}})
D = pd.DataFrame(D); display(D)
fig, ax = plt.subplots(figsize=(7, 5.5))
for c, colore in zip([60, 90], ["tab:blue", "tab:red"]):
    d = D[D["CHO [g]"] == c]
    ax.scatter(d["Δ TAR [%]"], d["Δ TBR [%]"], color=colore, label=f"{c} g")
    for _, r in d.iterrows(): ax.annotate(r.Paziente[-2:], (r["Δ TAR [%]"], r["Δ TBR [%]"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.axhline(0, color="k", lw=0.8); ax.axvline(0, color="k", lw=0.8); ax.grid(ls=":", alpha=0.5); ax.legend()
ax.set_xlabel("Δ TAR [punti %] (proposto − PID)"); ax.set_ylabel("Δ TBR [punti %] (proposto − PID)")
ax.set_title("In basso a destra: meno ipoglicemia al prezzo di più iperglicemia"); plt.tight_layout(); plt.show()

,Paziente,CHO [g],Δ TIR [%],Δ TAR [%],Δ TBR [%],Δ TBR tardivo [%],Δ Picco [mg/dL],Δ IAE fuori range [mg/dL*h]
0,adult#001,60,-16.8,16.8,0.0,0.0,17.9,41.0
1,adult#002,60,-1.0,1.0,0.0,0.0,1.3,8.3
2,adult#003,60,17.8,3.1,-20.9,-37.2,-23.8,-110.0
3,adult#004,60,10.6,-1.9,-8.8,-15.6,-22.9,-43.2
4,adult#005,60,-2.8,2.8,0.0,0.0,7.6,32.5
5,adult#006,60,16.9,3.8,-20.7,-36.7,-9.7,-149.8
6,adult#007,60,4.0,11.0,-15.0,-26.7,33.9,-23.4
7,adult#008,60,-2.8,14.4,-11.6,-20.6,14.3,14.8
8,adult#009,60,-4.7,4.7,0.0,0.0,8.4,52.9
9,adult#010,60,-3.7,3.8,0.0,0.0,4.2,44.4


## 3. Robustezza al rumore del sensore: 10 semi (60 g)

In [8]:
sk = [RC.key(p, 60, n, seed=s) for p in RC.PAZIENTI for n in ["PID", "PID cons.", "PID+IOB+PLGS"] for s in RC.SEMI]
RC.run_all(sk)
Sd = pd.DataFrame([{**RC.row(k), "Seme": k[3]} for k in sk])
display(RC.media_std(Sd, ["Paziente", "Controllore"],
        [RC.COLONNE[c] for c in ["TIR", "TAR", "TBR", "TBR<54", "TBR_tardivo", "picco", "IAE_range", "t_rientro", "n_commut"]]))

,Paziente,Controllore,TIR [%],TAR [%],TBR [%],TBR<54 [%],TBR tardivo [%],Picco [mg/dL],IAE fuori range [mg/dL*h],Rientro in range [h],Commutazioni vincolo [-],Non rientra [n]
0,adult#001,PID,93.7 ± 2.9,6.2 ± 2.8,0.1 ± 0.4,0.0 ± 0.0,0.2 ± 0.7,188.1 ± 4.9,6.3 ± 4.3,4.2 ± 1.6,0.0 ± 0.0,1
1,adult#001,PID cons.,83.9 ± 6.5,16.1 ± 6.5,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,199.6 ± 4.7,26.6 ± 14.3,5.1 ± 0.2,0.0 ± 0.0,0
2,adult#001,PID+IOB+PLGS,80.9 ± 6.7,19.1 ± 6.7,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,205.7 ± 7.8,43.7 ± 27.1,5.3 ± 0.3,71.4 ± 7.4,0
3,adult#002,PID,81.9 ± 1.0,18.1 ± 1.0,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,213.6 ± 3.7,62.1 ± 11.1,4.3 ± 0.2,0.0 ± 0.0,0
4,adult#002,PID cons.,80.5 ± 1.5,19.5 ± 1.5,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,214.5 ± 6.3,69.2 ± 18.1,4.4 ± 0.2,0.0 ± 0.0,0
5,adult#002,PID+IOB+PLGS,81.2 ± 1.9,18.8 ± 1.9,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,214.5 ± 8.8,69.1 ± 24.8,4.2 ± 0.2,54.1 ± 8.5,0
6,adult#003,PID,55.2 ± 5.8,21.9 ± 3.3,22.9 ± 2.7,11.0 ± 2.3,38.7 ± 2.6,247.0 ± 10.9,198.9 ± 42.1,9.9 ± 0.3,0.0 ± 0.0,0
7,adult#003,PID cons.,74.5 ± 3.5,25.5 ± 3.5,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,235.4 ± 12.6,134.6 ± 44.3,4.5 ± 0.5,0.0 ± 0.0,0
8,adult#003,PID+IOB+PLGS,79.8 ± 7.2,15.4 ± 5.1,4.7 ± 4.4,0.0 ± 0.0,8.4 ± 7.7,211.9 ± 14.1,54.7 ± 29.3,5.8 ± 3.3,61.8 ± 4.8,1


## 4. Lettura dei risultati (calcolata dalle tabelle)

In [9]:
for c in [60, 90]:
    d = D[D["CHO [g]"] == c]
    meno_ipo = d[d["Δ TBR [%]"] < 0]; piu_iper = d[d["Δ TAR [%]"] > 0]
    entrambi = d[(d["Δ TBR [%]"] < 0) & (d["Δ TAR [%]"] > 0)]
    print(f"{c} g: il proposto riduce il TBR in {len(meno_ipo)}/10 pazienti, aumenta il TAR in {len(piu_iper)}/10, "
          f"entrambe le cose (compromesso) in {len(entrambi)}/10; dTIR medio {d['Δ TIR [%]'].mean():+.1f} punti, "
          f"dTBR medio {d['Δ TBR [%]'].mean():+.1f}, dTAR medio {d['Δ TAR [%]'].mean():+.1f}")

60 g: il proposto riduce il TBR in 5/10 pazienti, aumenta il TAR in 9/10, entrambe le cose (compromesso) in 4/10; dTIR medio +1.7 punti, dTBR medio -7.7, dTAR medio +6.0
90 g: il proposto riduce il TBR in 7/10 pazienti, aumenta il TAR in 10/10, entrambe le cose (compromesso) in 7/10; dTIR medio +5.0 punti, dTBR medio -10.4, dTAR medio +5.4


**Limiti.** Simulazioni *in silico* con un solo pasto al giorno e guadagni non ritarati sui singoli pazienti:
i risultati mostrano il comportamento di parametri fissi su una popolazione eterogenea, non il miglior risultato
ottenibile per ciascun soggetto. Il controllore proposto **non** va presentato come uniformemente migliore: l'analisi
serve a mostrare dove e quanto il compromesso tra sicurezza e controllo dell'iperglicemia si sposta.